# Assignment 3 — A Degree-Planning Agent

Authored by Dr. Tiziana Ligorio for AI Agents - CSCI 395.32, Hunter College, CUNY.

> **Run this notebook locally.** If you have only used Google Colab so far, follow `SETUP.md` first; it takes about 15 minutes.

**Work in pairs. Submit one notebook and one results file per pair.**

**What you are building:** the **Riverview College degree-planning assistant**, a tool-using agent that answers students' questions about the Spring 2027 term: whether they can take a course, which sections are offered, whether two sections conflict, and how far they are from graduating.

In the LangChain ReAct tutorial, the tools and the evaluation test set were given to you. Here you take on more of the work. You write two of the agent's five tools and the agent's policy, design part of its test set, and improve the agent until it passes the tests.

The Riverview data is fictional and small on purpose. Every question has one correct answer that can be computed from the data, so you can tell exactly when the agent is right.

## Learning goals

By the end of the assignment, you should be able to:

- implement a tool as a function with a clear contract: its inputs, its outputs, and its errors;
- write an agent policy that decides when to use which tools, when the data does not have the answer, and when the student needs a person;
- evaluate an agent end to end on three checks: its answer, the tools it called, and the number of tool calls;
- design test cases that catch real failures; and
- improve an agent by reading its traces and changing one thing at a time.

## What you will do

1. Run the setup check and load the data (§1–2).
2. Read the helpers and the three provided tools (§3–4).
3. Write `check_eligibility` and `schedule_conflicts`, and pass the public tests (§5).
4. Read how the agent is built, then write its policy (§6–7).
5. Run the agent on the public cases. When a case fails, read its trace, change one thing in your policy, and rerun. Repeat until every public case passes, and record three of these changes in your iteration log (§8, §10).
6. Write six test cases of your own, and keep iterating until your agent passes them too (§9).
7. Save your results and disclose any AI assistance (§11–12).

The public tests and cases are a sample. Your tools and agent should work for any student, course, or question of the same kinds, so write general rules, not one per case.

Cells marked `# DO NOT MODIFY` are provided; do not change them. Write only in the cells marked `# YOUR WORK n`.

Each partner has an individual Understanding Check: be ready to explain any cell of your submission.

## Deliverables

- This notebook, with its **saved outputs**.
- `results.json`, which §11 writes.

Do not submit your `.env` file or your API key.

## Using AI assistants in this assignment

You may use an AI assistant as a learning partner, for example to explain a LangChain error or to discuss why a prompt rule did not take effect. You remain responsible for understanding and testing every cell you submit. Do not give an assistant this notebook and ask it to write your tools, policy, or cases. At the end, disclose what assistance you used and what you checked yourself.

## 1. Setup

Run the setup check first. It confirms that the notebook is using the project's Python, that the packages are installed, that the `data/` folder is next to the notebook, and that your OpenRouter key is in `.env`. If a check fails, its message names the `SETUP.md` step that fixes it.

Every agent in this course uses the same model, `openai/gpt-4.1-mini` at temperature 0 through OpenRouter, so results are comparable across teams.

In [18]:
# Setup check: run this cell first. If anything fails, follow SETUP.md.
import sys
from pathlib import Path

print("Python", sys.version.split()[0])
assert sys.version_info >= (3, 11), "This is not the project's Python. See SETUP.md, step 6 (select the .venv kernel)."
try:
    import dotenv, langchain, langchain_openai, pydantic  # noqa: F401
except ModuleNotFoundError as exc:
    raise SystemExit(f"Missing package: {exc.name}. See SETUP.md, steps 4 and 6.")
assert Path("data/catalog.json").exists(), "Data files not found. Open this notebook from the unzipped assignment folder (SETUP.md, step 2)."
dotenv.load_dotenv()
import os
assert os.getenv("OPENROUTER_API_KEY"), "OPENROUTER_API_KEY not found. Create your .env file (SETUP.md, step 5)."
print("Setup OK: packages, data files, and API key found.")

Python 3.12.15
Setup OK: packages, data files, and API key found.


In [19]:
# DO NOT MODIFY — imports
import json
import os
import re
from pathlib import Path
from typing import Literal, Union

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware, ToolCallLimitMiddleware, ToolRetryMiddleware
from langchain_core.messages import AIMessage, ToolMessage
from langchain_core.tools import StructuredTool
from langchain_openai import ChatOpenAI

_ = load_dotenv()

## 2. The Riverview data

The agent plans the **Spring 2027** term; the current term is Fall 2026, so courses in progress have no grade yet. The cell below loads four files from `data/`:

- `CATALOG`: courses by code, with credits, prerequisites, and the minimum prerequisite grade;
- `SECTIONS`: the Spring 2027 sections, with days, times, open seats, and modality;
- `STUDENTS`: eight student records by id;
- `REQUIREMENTS`: the degree and major rules.

**Read `data/README.md` before writing your tools.** It states the rules that every tool and every expected answer follows: grades and retakes, prerequisite groups, the order of the eligibility checks, what counts as a conflict, and degree progress. The printed course, CSCI 350, shows a prerequisite group: it needs CSCI 210 **and** either STAT 205 or MATH 250.

The student records are short. Read a few of them in `data/students.json`: knowing the data is how you write test cases with correct expected answers.

In [20]:
# DO NOT MODIFY — load the Riverview data
DATA_DIR = Path("data")
CATALOG = json.loads((DATA_DIR / "catalog.json").read_text(encoding="utf-8"))
SECTIONS = json.loads((DATA_DIR / "sections.json").read_text(encoding="utf-8"))["sections"]
STUDENTS = json.loads((DATA_DIR / "students.json").read_text(encoding="utf-8"))
REQUIREMENTS = json.loads((DATA_DIR / "requirements.json").read_text(encoding="utf-8"))

print(f"{len(CATALOG)} courses, {len(SECTIONS)} Spring 2027 sections, {len(STUDENTS)} students")
print(CATALOG["CSCI 350"])

16 courses, 26 Spring 2027 sections, 8 students
{'title': 'Machine Learning', 'credits': 3, 'prerequisites': [['CSCI 210'], ['STAT 205', 'MATH 250']], 'min_grade': 'C', 'offered': ['Spring']}


## 3. Helpers for your tools

The provided tools use four small helpers, and your tools can use them too. Using them keeps your tools consistent with the rules in `data/README.md`:

- `norm_course` accepts a course code the way a student might type it (`csci310`, `CSCI-310`) and returns `"CSCI 310"`, or `None` if the text is not a course code.
- `latest_grades` applies the retake rule: it keeps only the latest attempt of each course.
- `meets_min_grade` compares a grade with a minimum grade. `None` means any passing grade (D or better), and F and W never meet a minimum.
- `to_minutes` turns `"10:40"` into 640, so you can compare times as numbers.

The helpers do not decide eligibility or conflicts; that logic is yours.

In [21]:
# DO NOT MODIFY — helpers for your tools
GRADE_POINTS = {"A": 4.0, "A-": 3.7, "B+": 3.3, "B": 3.0, "B-": 2.7,
                "C+": 2.3, "C": 2.0, "C-": 1.7, "D": 1.0, "F": 0.0}
SEASONS = {"Spring": 0, "Summer": 1, "Fall": 2}


def norm_course(code):
    """'csci310', 'CSCI-310', ' csci 310 ' -> 'CSCI 310'. Returns None if the text is not a course code."""
    if not isinstance(code, str):
        return None
    match = re.fullmatch(r"\s*([A-Za-z]{2,5})[\s\-_]*(\d{3})\s*", code)
    return f"{match.group(1).upper()} {match.group(2)}" if match else None


def latest_grades(student):
    """{course: grade} for a student's completed courses, keeping only the latest attempt of each course."""
    latest = {}
    for record in student["completed"]:
        season, year = record["term"].split()
        when = (int(year), SEASONS[season])
        if record["course"] not in latest or when >= latest[record["course"]][0]:
            latest[record["course"]] = (when, record["grade"])
    return {course: grade for course, (when, grade) in latest.items()}


def meets_min_grade(grade, min_grade):
    """True if `grade` meets `min_grade`. With min_grade None, any passing grade (D or better) does. F and W never do."""
    if grade not in GRADE_POINTS:
        return False
    return GRADE_POINTS[grade] >= GRADE_POINTS[min_grade or "D"]


def to_minutes(hhmm):
    """'10:40' -> 640 (minutes after midnight)."""
    hours, minutes = hhmm.split(":")
    return int(hours) * 60 + int(minutes)


print(norm_course("csci310"), latest_grades(STUDENTS["S103"]), meets_min_grade("C-", "C"), to_minutes("10:40"))

CSCI 310 {'CSCI 110': 'B', 'ENG 101': 'B', 'MATH 120': 'B-'} False 640


## 4. The provided tools

The agent has five tools. Three are provided: `get_course`, `get_student_record`, and `find_sections`. Read them as examples of the contract your two tools must follow:

- **The docstring is the tool's description.** It is what the model reads when it decides whether and how to call the tool, so it says what the tool returns, not how it works.
- **The type hints become the argument schema** that the model fills in.
- **Expected problems are returned, not raised.** An unknown course or student comes back as `{"error": ..., "error_type": ...}`, so the agent can read the error and tell the student.

`find_sections` simulates a temporary outage, as in the tutorial: the first lookup of STAT 205 raises a `TimeoutError`, the way a real service fails. The retry middleware in §6 catches it and tries again. The runner restores the outage before every case, so each case faces it.

In [22]:
# DO NOT MODIFY — the three provided tools
SIMULATED_OUTAGES = {"STAT 205"}                  # the first section lookup of these courses times out
_outages_remaining = set(SIMULATED_OUTAGES)


def reset_simulated_outage():
    """Restore the simulated outage (the runner does this before every case)."""
    global _outages_remaining
    _outages_remaining = set(SIMULATED_OUTAGES)


def get_course(course_code: str) -> dict:
    """Look up a Riverview College course in the catalog.

    Returns its title, credits, prerequisites (each entry is one requirement; "STAT 205 or MATH 250" means
    either course meets it), the minimum grade required in prerequisites, and the terms it is usually offered.
    """
    code = norm_course(course_code)
    if code not in CATALOG:
        return {"error": f"No course '{course_code}' in the Riverview catalog.", "error_type": "unknown_course"}
    course = CATALOG[code]
    return {"course": code, "title": course["title"], "credits": course["credits"],
            "prerequisites": [" or ".join(group) for group in course["prerequisites"]],
            "min_grade": course["min_grade"], "offered": course["offered"]}


def get_student_record(student_id: str) -> dict:
    """Look up a Riverview College student's academic record.

    Returns the student's major, completed courses with grades and terms, Fall 2026 courses in progress,
    credits earned, credits remaining to the 120-credit minimum, GPA, and the major requirements not yet passed.
    """
    sid = student_id.strip().upper() if isinstance(student_id, str) else None
    if sid not in STUDENTS:
        return {"error": f"No student '{student_id}' in the records.", "error_type": "unknown_student"}
    student = STUDENTS[sid]
    grades = latest_grades(student)
    passed = {c for c, g in grades.items() if meets_min_grade(g, None)}
    graded = {c: g for c, g in grades.items() if g in GRADE_POINTS}
    credits_earned = sum(CATALOG[c]["credits"] for c in passed)
    attempted = sum(CATALOG[c]["credits"] for c in graded)
    gpa = round(sum(GRADE_POINTS[g] * CATALOG[c]["credits"] for c, g in graded.items()) / attempted, 2) if attempted else None
    major = REQUIREMENTS["majors"].get(student["major"])
    remaining = None
    if major:
        remaining = [c for c in major["required"] if c not in passed]
        remaining += [" or ".join(g) for g in major["choose_one"] if not passed.intersection(g)]
    return {"student_id": sid, "name": student["name"], "major": student["major"],
            "completed": student["completed"],
            "in_progress": [r["course"] for r in student["in_progress"]],
            "credits_earned": credits_earned,
            "credits_remaining": max(0, REQUIREMENTS["general"]["min_total_credits"] - credits_earned),
            "gpa": gpa, "remaining_major_requirements": remaining}


def find_sections(course_code: str) -> dict:
    """List the Spring 2027 sections of a course.

    Returns each section's id, meeting days, start and end times, open seats, and modality
    (in_person, online_sync, or online_async). An empty list means the course has no Spring 2027 sections.
    """
    code = norm_course(course_code)
    if code not in CATALOG:
        return {"error": f"No course '{course_code}' in the Riverview catalog.", "error_type": "unknown_course"}
    if code in _outages_remaining:
        _outages_remaining.discard(code)
        raise TimeoutError("The section service timed out. Try again.")
    return {"course": code, "term": "Spring 2027", "sections": [s for s in SECTIONS if s["course"] == code]}


print(get_course("csci 350"))

{'course': 'CSCI 350', 'title': 'Machine Learning', 'credits': 3, 'prerequisites': ['CSCI 210', 'STAT 205 or MATH 250'], 'min_grade': 'C', 'offered': ['Spring']}


## 5. Your tools

Write `check_eligibility` (YOUR WORK 1) and `schedule_conflicts` (YOUR WORK 2). The docstrings are given, and they are the contract: the keys and values to return, and the error to return for each kind of bad input. Do not change the function names, arguments, or docstrings; the model reads the docstrings, and the rest of the notebook calls your functions with these arguments.

- Follow the rules in `data/README.md` exactly, including the order of the eligibility checks.
- Read the data directly (`CATALOG`, `SECTIONS`, `STUDENTS`) and use the helpers. Do not call `find_sections` from your tools: it is subject to the outage.
- Keep each tool, including any helper functions you add, inside its own cell.

The public tests below include one or two cases of each kind. Passing them is necessary but not enough: try your tools on other students, courses, and section ids, and check the results against the data by hand.

In [23]:
# YOUR WORK 1 — check_eligibility
# Keep all your code for this tool, including any helper functions you add, inside this cell.
def check_eligibility(student_id: str, course_code: str) -> dict:
    """Check whether a Riverview College student meets the prerequisites for a course in Spring 2027.

    Returns {"status": "eligible" | "not_eligible" | "pending" | "already_taken",
             "missing": [prerequisites not met by completed courses, e.g. "CSCI 210" or "STAT 205 or MATH 250"],
             "reason": "one sentence"}.
    "pending" means the student will be eligible only if they pass a prerequisite they are taking now.
    For an unknown student or course: {"error": "...", "error_type": "unknown_student" | "unknown_course"}.
    """
    # 1. Normalize and validate the inputs.
    sid = student_id.strip().upper() if isinstance(student_id, str) else None
    code = norm_course(course_code)

    if sid not in STUDENTS:
        return {
            "error": f"No student '{student_id}' in the records.",
            "error_type": "unknown_student"
        }

    if code not in CATALOG:
        return {
            "error": f"No course '{course_code}' in the Riverview catalog.",
            "error_type": "unknown_course"
        }

    student = STUDENTS[sid]
    course = CATALOG[code]

    # 2. Apply the first eligibility rule: already taken.
    grades = latest_grades(student)
    passed_courses = {
        c for c, grade in grades.items()
        if meets_min_grade(grade, None)
    }
    in_progress_courses = {
        norm_course(record["course"])
        for record in student["in_progress"]
    }

    if code in passed_courses or code in in_progress_courses:
        return {
            "status": "already_taken",
            "missing": [],
            "reason": (
                f"{code} has already been passed or is currently in progress."
            )
        }

    # 3. Evaluate every prerequisite group.
    missing = []
    pending_groups = []

    for group in course["prerequisites"]:
        # A group is met if ANY course in it satisfies the grade requirement.
        completed_group = any(
            prereq in grades
            and meets_min_grade(grades[prereq], course["min_grade"])
            for prereq in group
        )

        if completed_group:
            continue

        # If not completed, check whether ANY alternative is in progress.
        in_progress_group = any(
            prereq in in_progress_courses
            for prereq in group
        )

        group_description = " or ".join(group)

        if in_progress_group:
            pending_groups.append(group_description)
        else:
            missing.append(group_description)

    # 4. Apply the remaining eligibility rules in the specified order.
    if missing:
        return {
            "status": "not_eligible",
            "missing": missing,
            "reason": (
                "The student has not completed or is not currently taking "
                "all required prerequisite groups."
            )
        }

    if pending_groups:
        return {
            "status": "pending",
            "missing": pending_groups,
            "reason": (
                "The student must pass the in-progress prerequisite "
                "coursework to be eligible."
            )
        }

    return {
        "status": "eligible",
        "missing": [],
        "reason": "All prerequisite groups are satisfied by completed coursework."
    }


In [24]:
# YOUR WORK 2 — schedule_conflicts
# Keep all your code for this tool, including any helper functions you add, inside this cell.
def schedule_conflicts(section_ids: list[str]) -> dict:
    """Check a list of Spring 2027 section ids for time conflicts.

    Returns {"conflicts": [[id_a, id_b], ...] (each pair sorted, the list sorted; [] if none),
             "ok": True if there are no conflicts}.
    For fewer than two ids or an unknown id: {"error": "...", "error_type": "invalid_input" | "unknown_section"}.
    """
     # 1. Validate the input.
    if not isinstance(section_ids, list) or len(section_ids) < 2:
        return {
            "error": "Provide at least two section IDs.",
            "error_type": "invalid_input"
        }

    # 2. Build a lookup from section ID to section information.
    sections_by_id = {
        section["section_id"]: section
        for section in SECTIONS
    }

    # 3. Validate that all requested section IDs exist.
    for section_id in section_ids:
        if section_id not in sections_by_id:
            return {
                "error": f"Unknown section ID '{section_id}'.",
                "error_type": "unknown_section"
            }

    # 4. Compare every unique pair.
    conflicts = []
    ids = sorted(set(section_ids))

    for i in range(len(ids)):
        for j in range(i + 1, len(ids)):
            id_a = ids[i]
            id_b = ids[j]

            section_a = sections_by_id[id_a]
            section_b = sections_by_id[id_b]

            # Asynchronous online sections have no scheduled meeting time.
            if section_a["modality"] == "online_async":
                continue
            if section_b["modality"] == "online_async":
                continue

            # Sections must share at least one meeting day.
            shared_days = set(section_a["days"]) & set(section_b["days"])
            if not shared_days:
                continue

            # Convert start/end times to minutes after midnight.
            start_a = to_minutes(section_a["start"])
            end_a = to_minutes(section_a["end"])
            start_b = to_minutes(section_b["start"])
            end_b = to_minutes(section_b["end"])

            # Strict inequalities mean touching endpoints do not conflict.
            if start_a < end_b and start_b < end_a:
                conflicts.append([id_a, id_b])

    # 5. Return sorted conflict pairs.
    conflicts.sort()

    return {
        "conflicts": conflicts,
        "ok": len(conflicts) == 0
    }

In [25]:
# DO NOT MODIFY — public tests for your two tools
# One or two cases of each kind.
PUBLIC_TESTS = [
    ("eligible", lambda: check_eligibility("S101", "CSCI 310")["status"] == "eligible"),
    ("minimum grade", lambda: check_eligibility("S102", "CSCI 310")["missing"] == ["CSCI 210"]
                              and check_eligibility("S102", "CSCI 310")["status"] == "not_eligible"),
    ("prerequisite in progress", lambda: check_eligibility("S104", "CSCI 310")["status"] == "pending"),
    ("either-or prerequisite", lambda: check_eligibility("S106", "CSCI 350")["status"] == "eligible"),
    ("already taken", lambda: check_eligibility("S106", "CSCI 310")["status"] == "already_taken"),
    ("unknown student", lambda: check_eligibility("S999", "CSCI 310")["error_type"] == "unknown_student"),
    ("overlap", lambda: schedule_conflicts(["CSCI310-02", "CSCI340-01"])
                        == {"conflicts": [["CSCI310-02", "CSCI340-01"]], "ok": False}),
    ("back-to-back", lambda: schedule_conflicts(["CSCI310-01", "MATH220-01"]) == {"conflicts": [], "ok": True}),
    ("online asynchronous", lambda: schedule_conflicts(["PHIL120-W1", "CSCI310-01"])["ok"] is True),
    ("fewer than two ids", lambda: schedule_conflicts(["CSCI310-01"])["error_type"] == "invalid_input"),
]

passed = 0
for name, test in PUBLIC_TESTS:
    try:
        ok = bool(test())
    except Exception as exc:
        ok, name = False, f"{name} (raised {type(exc).__name__}: {exc})"
    passed += ok
    print(f"{'pass' if ok else 'FAIL'}  {name}")
print(f"{passed}/{len(PUBLIC_TESTS)} public tests passed")

pass  eligible
pass  minimum grade
pass  prerequisite in progress
pass  either-or prerequisite
pass  already taken
pass  unknown student
pass  overlap
pass  back-to-back
pass  online asynchronous
pass  fewer than two ids
10/10 public tests passed


## 6. The agent

The agent is LangChain's `create_agent`, as in the tutorial. The two cells below are provided.

**The answer format.** `AgentAnswer` is a Pydantic class, like the tutorial's `Verdict`. Passing it as `response_format` makes the agent's final answer an `AgentAnswer` object instead of free text, so the checks compare fields, not wording:

- `status`: `answered`, `not_available`, or `needs_person`;
- `value`: the answer itself, in the format its description gives (for example `"pending"`, a list of section ids, or a number), and null unless the status is `answered`;
- `explanation`: one to three sentences for the student.

The field descriptions are sent to the model, so they are part of what it reads.

**Building the agent.** `build_agent(config)` assembles the model, the five tools, your policy, and three middleware guardrails from the tutorial. The config is a dictionary:

| Key | Default | What it controls |
|---|---|---|
| `system_prompt` | one sentence | your policy (§7) |
| `tool_descriptions` | `{}` | optional replacement descriptions, by tool name |
| `model_call_limit` | 8 | model calls per question before the run stops |
| `tool_call_limit` | 10 | tool calls per question before the run stops |
| `tool_max_retries` | 2 | retries after a tool raises an exception, such as the outage |

The model and temperature are fixed. `check_config` rejects unknown keys, so a typo raises an error instead of silently changing nothing.

**Tool descriptions are part of your policy.** The model reads a tool's description every time it decides whether to call that tool, and with which arguments. A rule about one tool, such as which arguments are valid, can go in that tool's description instead of the system prompt. Extend the docstring rather than replacing it, for example `check_eligibility.__doc__ + "\n    Your rule."`.

**`ask(agent, question)`** runs one question and returns a record: the answer, the tool calls, a step-by-step trace (thought, action, observation, answer), and any error. A run that reaches a call limit has no answer, and the record says so.

`build_agent` uses the current version of each tool, so after you change a tool, rerun YOUR WORK 3 to rebuild the agent.

In [19]:
# DO NOT MODIFY — the answer format
class AgentAnswer(BaseModel):
    """The agent's final answer to a student's question."""

    status: Literal["answered", "not_available", "needs_person"] = Field(
        description="'answered' if the tools gave the answer; 'not_available' if the answer cannot be found: the "
                    "Riverview data does not contain it, or the question lacks something only the student can give "
                    "(such as their student id); 'needs_person' if only a person can decide (a waiver, an exception, "
                    "a registration).")
    value: Union[str, float, list[str], list[list[str]], None] = Field(
        description="The answer itself, null unless status is 'answered'. Eligibility: one of eligible, "
                    "not_eligible, pending, already_taken. Sections: a list of section ids. Conflicts: a list of "
                    "conflicting pairs of section ids, [] if none. Credits or GPA: a number.")
    explanation: str = Field(description="One to three sentences for the student.")

In [27]:
# DO NOT MODIFY — build the agent from a config
MODEL = "openai/gpt-4.1-mini"
TEMPERATURE = 0
DEFAULT_CONFIG = {
    "system_prompt": "You are the Riverview College degree-planning assistant.",
    "tool_descriptions": {},           # optional: {tool name: a better description than its docstring}
    "model_call_limit": 8,
    "tool_call_limit": 10,
    "tool_max_retries": 2,
}
TOOL_NAMES = ["get_course", "get_student_record", "find_sections", "check_eligibility", "schedule_conflicts"]


def make_model():
    return ChatOpenAI(model=MODEL, temperature=TEMPERATURE, base_url="https://openrouter.ai/api/v1",
                      api_key=os.environ["OPENROUTER_API_KEY"], max_tokens=2048,)


def check_config(config):
    """Reject unknown keys, so a typo cannot silently change nothing."""
    unknown = set(config) - set(DEFAULT_CONFIG)
    if unknown:
        raise ValueError(f"Unknown config keys: {sorted(unknown)}. Allowed: {sorted(DEFAULT_CONFIG)}")
    bad = set(config.get("tool_descriptions", {})) - set(TOOL_NAMES)
    if bad:
        raise ValueError(f"tool_descriptions names unknown tools: {sorted(bad)}")
    return {**DEFAULT_CONFIG, **config}


def build_agent(config):
    """The model, the five tools, your policy and guardrails, and AgentAnswer as the answer format."""
    config = check_config(config)
    functions = {name: globals()[name] for name in TOOL_NAMES}     # the latest version of each function
    tools = [StructuredTool.from_function(func=functions[name], name=name,
                                          description=config["tool_descriptions"].get(name) or functions[name].__doc__)
             for name in TOOL_NAMES]
    return create_agent(
        model=make_model(),
        tools=tools,
        system_prompt=config["system_prompt"],
        middleware=[
            ModelCallLimitMiddleware(run_limit=config["model_call_limit"]),
            ToolCallLimitMiddleware(run_limit=config["tool_call_limit"]),
            ToolRetryMiddleware(max_retries=config["tool_max_retries"], initial_delay=0.5, jitter=False),
        ],
        response_format=AgentAnswer,
    )


def ask(agent, question):
    """Run one question. Returns a record: the answer, the tool calls read from the messages, a trace, any error."""
    record = {"question": question, "answer": None, "tool_calls": [], "tools_used": [], "steps": [], "error": None}
    try:
        result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    except Exception as exc:
        record["error"] = f"{type(exc).__name__}: {exc}"
        return record
    for message in result["messages"]:
        if isinstance(message, AIMessage):
            if message.tool_calls and isinstance(message.content, str) and message.content:
                record["steps"].append({"step": "thought", "text": message.content})
            for call in message.tool_calls:
                record["tool_calls"].append({"name": call["name"], "args": call["args"]})
                record["steps"].append({"step": "action", "tool": call["name"], "args": call["args"]})
        elif isinstance(message, ToolMessage):
            record["steps"].append({"step": "observation", "tool": message.name, "text": str(message.content)})
    record["tools_used"] = [call["name"] for call in record["tool_calls"]]
    answer = result.get("structured_response")
    if isinstance(answer, AgentAnswer):
        record["answer"] = answer.model_dump()
        record["steps"].append({"step": "answer", "text": json.dumps(record["answer"])})
    else:
        record["error"] = "The agent stopped without an answer (for example, it reached a call limit)."
    return record

## 7. Your agent's policy

Write your agent's policy as the system prompt in YOUR WORK 3, plus any tool descriptions you want to change. Your policy must handle each of these, and the public cases show whether it does:

* using the tools for every fact;
* questions that lack information the agent needs, such as the student's id;
* the difference between information the data does not have (`not_available`) and decisions or actions only a person can take (`needs_person`), such as waivers, overrides, and registering;
* calling only the tools a question needs.

Write general rules about kinds of questions, not one rule per public case: students word their questions in many ways and ask about other students and courses. Keep each rule short, and tied to a behavior you can check.

Avoid concrete examples in your rules, such as a student id or a course. The model can copy an example into an answer where it does not belong, for example using an id from your prompt when the student gave none.

In [28]:
# YOUR WORK 3 — your agent's policy and guardrails
CONFIG = {
    **DEFAULT_CONFIG,
     "system_prompt": """You are the Riverview College degree-planning assistant.
        Use the available tools for all factual claims about Riverview courses, students, prerequisites, sections, schedules, credits, GPA, and degree requirements. Treat tool results as the source of truth; never invent or assume facts that the tools have not established.
        For schedule conflict questions, if the user provides two or more exact section IDs, call schedule_conflicts directly with those IDs. Do not look up the sections or require a student ID. If the user provides course codes without section IDs, use find_sections to identify the relevant sections before checking conflicts.
        For schedule conflict questions, set the answer's `value` to the `conflicts` field returned by schedule_conflicts, preserving its exact nested-list structure. Each conflict is a pair of section IDs, so return a list of pairs (for example, [["CSCI320-01", "ENG210-01"]]) or an empty list ([]) when there are no conflicts. Never flatten the list of pairs into a single list of section IDs.
        Call only the tools necessary to answer the student's question. Use multiple tools when the question requires combining information from different sources.
        If required information is missing, ask the student to provide it when appropriate and return `not_available` with a null value. Do not guess a student's identity or substitute an assumed course or section.
        Use `not_available` when the answer cannot be established from the available data. Use needs_person when the request requires a human decision or action that the tools cannot perform, including exceptions, waivers, overrides, or registration.
        Return `answered` only when the tool results support the answer. Set value to the requested result using the appropriate data type, and set it to null for all other statuses. Keep the explanation accurate, concise, and between one and three sentences.""",
}
assert "TODO" not in CONFIG["system_prompt"], "Write your system prompt."
agent = build_agent(CONFIG)

## 8. Evaluate

The checks and the runner are provided. `run_cases(agent, cases)` asks every question, restores the outage before each one, and checks each record three ways:

- **answer**: the status matches and, for an answered question, so does the value (codes cleaned up, list order ignored, numbers within 0.01);
- **tools**: every expected tool was called, and no tool outside the expected and allowed ones;
- **budget**: the number of tool calls is at most the case's `max_tool_calls`.

`show_results` prints one row per case, with a note naming any missing or unnecessary tools. The cell below runs the 11 public cases.

**Iterate until every public case passes.** For each failure:

1. **Read the trace before changing the prompt.** `print_trace(record)` shows the question, each tool call with its arguments, each observation, and the answer. To get a record by id, use `next(r for r in public_records if r["id"] == "waiver_csci310")`. Find the first step that went wrong.
2. **Change one thing in your policy**, either the system prompt or a tool description, and rerun YOUR WORK 3.
3. **Rerun all the public cases**, not only the one you fixed: a rule that fixes one case can break another.

Record three of these changes in your iteration log (§10) as you go. While working on one case you can run it alone, for example `run_cases(agent, [c for c in PUBLIC_CASES if c["id"] == "waiver_csci310"])`, but rerun the whole set before moving on.

Keep the tutorial's cautions in mind:

* **One change at a time.** If you change two rules and the case passes, you do not know which one mattered.
* **Run again before concluding.** Model answers vary, and a case that flips once may be noise.
* **Do not tune to the public cases.** A rule that repeats a public question's wording will not help with the next question. Ask what kind of question failed, and write the rule for that kind.

Raising `tool_call_limit` or `model_call_limit` does not fix a budget failure. Each case's budget is far below those limits, which exist only to stop a runaway agent.

In [29]:
# DO NOT MODIFY — checks and runner
CATEGORIES = ["not_available", "needs_person", "eligibility", "conflicts", "multi_step", "outage"]
CASE_KEYS = {"id", "category", "question", "fields", "expected_status", "expected_value",
             "expected_tools", "max_tool_calls"}


def normalize_value(value):
    """Make answers comparable: codes cleaned up, lists sorted, pair order ignored, numbers as floats."""
    def text(x):
        s = str(x).strip()
        return norm_course(s) or (s.upper() if re.fullmatch(r"[A-Za-z]+\d+-\w+", s) else s.lower().replace(" ", "_"))
    if value is None or isinstance(value, bool):
        return value
    if isinstance(value, (int, float)):
        return float(value)
    if isinstance(value, str):
        try:
            return float(value)
        except ValueError:
            return text(value)
    if value and all(isinstance(v, (list, tuple)) for v in value):
        return sorted(sorted(text(x) for x in pair) for pair in value)
    return sorted(text(v) for v in value)


def check_answer(case, record, tolerance=0.01):
    """The status matches, and for answered questions the value matches too."""
    answer = record["answer"]
    if not answer or answer["status"] != case["expected_status"]:
        return False
    if case["expected_status"] != "answered":
        return True
    expected, got = normalize_value(case["expected_value"]), normalize_value(answer["value"])
    if isinstance(expected, float) and isinstance(got, float):
        return abs(expected - got) <= tolerance
    return expected == got


def check_tools(case, record):
    """Every expected tool was used, and no tool outside expected_tools + allowed_tools."""
    used, expected = set(record["tools_used"]), set(case["expected_tools"])
    missing, extra = expected - used, used - expected - set(case.get("allowed_tools", []))
    note = (f"missing {sorted(missing)} " if missing else "") + (f"unnecessary {sorted(extra)}" if extra else "")
    return not missing and not extra, note.strip()


def check_budget(case, record):
    return len(record["tool_calls"]) <= case["max_tool_calls"]


def validate_cases(cases):
    """Catch format mistakes early. It does not check that the expected answers are right."""
    problems = []
    for i, case in enumerate(cases):
        cid = case.get("id", f"case {i}")
        if CASE_KEYS - set(case):
            problems.append(f"{cid}: missing {sorted(CASE_KEYS - set(case))}")
        if set(case) - CASE_KEYS - {"allowed_tools"}:
            problems.append(f"{cid}: unknown keys {sorted(set(case) - CASE_KEYS - {'allowed_tools'})}")
        if case.get("expected_status") not in ("answered", "not_available", "needs_person"):
            problems.append(f"{cid}: expected_status must be answered, not_available, or needs_person")
    if sorted(c.get("category", "") for c in cases) != sorted(CATEGORIES):
        problems.append(f"write exactly one case for each category: {CATEGORIES}")
    if problems:
        raise ValueError("Case problems:\n  " + "\n  ".join(problems))
    print(f"{len(cases)} cases look well formed.")


def run_cases(agent, cases):
    records = []
    for case in cases:
        reset_simulated_outage()                       # every case starts with the outage in place
        record = ask(agent, case["question"])
        tools_ok, note = check_tools(case, record)
        record.update({"id": case["id"], "category": case["category"], "tools_note": note,
                       "checks": {"answer": check_answer(case, record), "tools": tools_ok,
                                  "budget": check_budget(case, record)}})
        records.append(record)
        print(".", end="", flush=True)
    print()
    return records


def count_passes(records):
    return {check: sum(r["checks"][check] for r in records) for check in ("answer", "tools", "budget")}


def show_results(records):
    mark = lambda ok: "pass" if ok else "FAIL"
    print(f"{'case':<26}{'category':<15}{'answer':<8}{'tools':<8}{'budget':<8}{'calls':<7}note")
    for r in records:
        c = r["checks"]
        print(f"{r['id']:<26}{r['category']:<15}{mark(c['answer']):<8}{mark(c['tools']):<8}"
              f"{mark(c['budget']):<8}{len(r['tool_calls']):<7}{r['error'] or r['tools_note']}")
    for check, passed in count_passes(records).items():
        print(f"{check:>7}: {passed}/{len(records)} passed")


def print_trace(record):
    labels = {"thought": "Thought", "action": "Action", "observation": "Observation", "answer": "Answer"}
    print(f"{'Question:':<14}{record['question']}")
    for s in record["steps"]:
        detail = f"{s['tool']}({s['args']})" if s["step"] == "action" else s["text"]
        print(f"{labels[s['step']] + ':':<14}{detail}")
    if record["error"]:
        print(f"{'Error:':<14}{record['error']}")


def save_results(path, **blocks):
    """Write the results file, leaving out the step-by-step traces to keep it small."""
    def trim(obj):
        if isinstance(obj, dict):
            return {k: trim(v) for k, v in obj.items() if k != "steps"}
        if isinstance(obj, list):
            return [trim(v) for v in obj]
        return obj
    Path(path).write_text(json.dumps(trim(blocks), indent=2, default=str), encoding="utf-8")
    print(f"Saved {path}")

In [30]:
PUBLIC_CASES = json.loads((DATA_DIR / "public_cases.json").read_text(encoding="utf-8"))
public_records = run_cases(agent, PUBLIC_CASES)
show_results(public_records)

...........
case                      category       answer  tools   budget  calls  note
elig_s101_csci310         eligibility    pass    pass    pass    1      
elig_s102_csci310         eligibility    pass    pass    pass    1      
conflict_back_to_back     conflicts      pass    pass    pass    1      
sections_csci210          sections       pass    pass    pass    1      
credits_remaining_s106    records        pass    pass    pass    1      
gpa_s101                  records        pass    pass    pass    1      
instructor_csci310        not_available  pass    pass    pass    0      
waiver_csci310            needs_person   pass    pass    pass    0      
no_student_id_gpa         not_available  pass    pass    pass    0      
courses_csci320_eng210    multi_step     pass    pass    pass    3      
outage_stat205_sections   outage         pass    pass    pass    1      
 answer: 11/11 passed
  tools: 11/11 passed
 budget: 11/11 passed


In [31]:
record = next(
    r for r in public_records
    if r["id"] == "conflict_back_to_back"
)
print_trace(record)

Question:     Do sections CSCI310-01 and MATH220-01 conflict?
Action:       schedule_conflicts({'section_ids': ['CSCI310-01', 'MATH220-01']})
Observation:  {"conflicts": [], "ok": true}
Answer:       {"status": "answered", "value": [], "explanation": "The sections CSCI310-01 and MATH220-01 do not have any scheduling conflicts. You can take both without overlapping class times."}


In [32]:
for case in PUBLIC_CASES:
    print({
        "id": case["id"],
        "category": case["category"],
        "question": case["question"],
        "fields": case["fields"],
        "expected_status": case["expected_status"],
        "expected_value": case["expected_value"],
        "expected_tools": case["expected_tools"],
        "max_tool_calls": case["max_tool_calls"],
    })

{'id': 'elig_s101_csci310', 'category': 'eligibility', 'question': 'Can student S101 take CSCI 310 in Spring 2027?', 'fields': {'intent': 'eligibility', 'student_id': 'S101', 'course': 'CSCI 310'}, 'expected_status': 'answered', 'expected_value': 'eligible', 'expected_tools': ['check_eligibility'], 'max_tool_calls': 2}
{'id': 'elig_s102_csci310', 'category': 'eligibility', 'question': "I'm S102. Am I eligible for CSCI 310 next term?", 'fields': {'intent': 'eligibility', 'student_id': 'S102', 'course': 'CSCI 310'}, 'expected_status': 'answered', 'expected_value': 'not_eligible', 'expected_tools': ['check_eligibility'], 'max_tool_calls': 2}
{'id': 'conflict_back_to_back', 'category': 'conflicts', 'question': 'Do sections CSCI310-01 and MATH220-01 conflict?', 'fields': {'intent': 'conflicts', 'section_ids': ['CSCI310-01', 'MATH220-01']}, 'expected_status': 'answered', 'expected_value': [], 'expected_tools': ['schedule_conflicts'], 'max_tool_calls': 1}
{'id': 'sections_csci210', 'category'

## 9. Your test cases

Write six test cases, one per category, in YOUR WORK 4. Each category is a kind of failure your agent could have. Write each case so that it would fail if your agent had that failure:

| Category | The case should fail if your agent... |
|---|---|
| `not_available` | makes up an answer the data does not have |
| `needs_person` | decides something only a person can |
| `eligibility` | judges eligibility itself instead of calling `check_eligibility`, or applies a wrong rule |
| `conflicts` | applies a wrong conflict rule, for example counting sections that only touch as a conflict |
| `multi_step` | answers without finishing a two-step task: finding the sections of two courses, then checking them |
| `outage` | gives up on a temporary failure instead of retrying (the question must need STAT 205 sections) |

**The `fields`** state, in a structured form, what the question asks. Work out the expected answer from them:

| `intent` | Other fields | Expected value |
|---|---|---|
| `eligibility` | `student_id`, `course` | `eligible`, `not_eligible`, `pending`, or `already_taken` |
| `conflicts` | `section_ids` | list of conflicting pairs, `[]` if none |
| `sections`, `online_sections`, `open_sections` | `course` | list of section ids |
| `prerequisites` | `course` | list of requirements, like `"STAT 205 or MATH 250"` |
| `course_credits` | `course` | a number |
| `credits_earned`, `credits_remaining`, `gpa` | `student_id` | a number |
| `course_conflicts` | `courses`: two courses that each have only one Spring 2027 section, so the question has a single answer | list of conflicting pairs |
| `not_available`, `needs_person` | none | `null` |

The public cases in `data/public_cases.json` show an example of most of these.

**The other keys:**

- `expected_status` and `expected_value`: work them out by hand from the data and the rules in `data/README.md`.
- `expected_tools`: the tools a correct agent must call.
- `allowed_tools` (optional): tools a correct agent may also call.
- `max_tool_calls`: the most calls a careful agent needs; the public cases show typical budgets.

A correct agent must pass your case. A case that fails a correct agent is a false alarm, so do not require a tool the agent does not need, or a budget below what it needs. `validate_cases` checks only the format. When your agent fails one of your cases, check the case first (is the expected value right?), then your policy.

In [33]:
# YOUR WORK 4 — your six test cases, one per category

MY_CASES = [
    {
        "id": "my_not_available_room",
        "category": "not_available",
        "question": "Which room is CSCI320-01 held in?",
        "fields": {"intent": "not_available"},
        "expected_status": "not_available",
        "expected_value": None,
        "expected_tools": [],
        "allowed_tools": ["find_sections", "get_course"],
        "max_tool_calls": 2
    },
    {
        "id": "my_needs_person_grade_change",
        "category": "needs_person",
        "question": (
            "I'm S107 and I think my C- in CSCI 110 was graded unfairly. "
            "Please change it to a C so I can take CSCI 210."
        ),
        "fields": {"intent": "needs_person"},
        "expected_status": "needs_person",
        "expected_value": None,
        "expected_tools": [],
        "max_tool_calls": 3
    },
    {
        "id": "my_elig_s105_csci310",
        "category": "eligibility",
        "question": "I'm S105. Can I take CSCI 310 next term?",
        "fields": {
            "intent": "eligibility",
            "student_id": "S105",
            "course": "CSCI 310"
        },
        "expected_status": "answered",
        "expected_value": "not_eligible",
        "expected_tools": ["check_eligibility"],
        "max_tool_calls": 2
    },
    {
        "id": "my_conflicts_touch_and_overlap",
        "category": "conflicts",
        "question": (
            "Do any of CSCI110-01, MATH220-01 and CSCI340-02 conflict?"
        ),
        "fields": {
            "intent": "conflicts",
            "section_ids": [
                "CSCI110-01",
                "MATH220-01",
                "CSCI340-02"
            ]
        },
        "expected_status": "answered",
        "expected_value": [
            ["CSCI340-02", "MATH220-01"]
        ],
        "expected_tools": ["schedule_conflicts"],
        "max_tool_calls": 1
    },
    {
        "id": "my_multi_step_math250_csci350",
        "category": "multi_step",
        "question": (
            "Do the Spring 2027 sections of MATH 250 and CSCI 350 conflict?"
        ),
        "fields": {
            "intent": "course_conflicts",
            "courses": ["MATH 250", "CSCI 350"]
        },
        "expected_status": "answered",
        "expected_value": [],
        "expected_tools": ["find_sections", "schedule_conflicts"],
        "max_tool_calls": 3
    },
    {
        "id": "my_outage_stat205_online",
        "category": "outage",
        "question": "Which STAT 205 sections are online?",
        "fields": {
            "intent": "online_sections",
            "course": "STAT 205"
        },
        "expected_status": "answered",
        "expected_value": ["STAT205-W1"],
        "expected_tools": ["find_sections"],
        "max_tool_calls": 1
    }
]

validate_cases(MY_CASES)
my_records = run_cases(agent, MY_CASES)
show_results(my_records)

6 cases look well formed.
......
case                      category       answer  tools   budget  calls  note
my_not_available_room     not_available  pass    pass    pass    0      
my_needs_person_grade_changeneeds_person   pass    pass    pass    0      
my_elig_s105_csci310      eligibility    pass    pass    pass    1      
my_conflicts_touch_and_overlapconflicts      pass    pass    pass    1      
my_multi_step_math250_csci350multi_step     pass    pass    pass    3      
my_outage_stat205_online  outage         pass    pass    pass    1      
 answer: 6/6 passed
  tools: 6/6 passed
 budget: 6/6 passed


## 10. Your iteration log

Write the log as you iterate, not at the end. Pick three changes that taught you something; one can be a change that did not work, or one that broke another case. Be specific: name the case, describe the step in the trace that went wrong, and give the rule you added or changed. Your Understanding Check starts from this log, so each partner should be able to explain every entry.

**YOUR WORK 5:** record three of the changes you made while getting your agent to pass the public cases and your own. Replace each TODO.

**Change 1**
- **Failing case:** my_not_available_room, tools check.
- **What the trace showed:** The agent called find_sections for CSCI320-01 before answering that the room isn’t in the data, and the case allowed no tool calls.
- **What you changed:** The case, not the policy. I added allowed_tools: ["find_sections"], because checking a section’s fields before saying a room isn’t recorded is reasonable. The case still fails if the agent names a room.
- **Result on the rerun:** all six cases passed, but the agent made 0 calls that run, so the change was not exercised. I’m treating it as unverified.

**Change 2**
- **Failing case:** conflict_back_to_back, budget check
- **What the trace showed:** The agent called find_sections for both courses before calling schedule_conflicts, even though the question already provided both exact section IDs. This used 3 tool calls when the case allowed only 1.
- **What you changed:** Clarified the system prompt to state that schedule conflict checks do not require a student ID and that, when exact section IDs are provided, the agent should call schedule_conflicts directly without looking up the sections.
- **Result on the rerun:** conflict_back_to_back passed all three checks. The agent called schedule_conflicts exactly once, returned the correct empty conflict list ([]), and the public budget check improved from 10/11 to 11/11. However, courses_csci320_eng210, which had passed in the baseline, then failed its answer check. I did not establish whether my new rule caused this or it was run-to-run variation; the trace showed an output-format problem, which I fixed in Change 3.

**Change 3**
- **Failing case:** courses_csci320_eng210, answer check.
- **What the trace showed:** The agent correctly looked up both courses, called schedule_conflicts, and identified the conflict. However, it flattened the tool's nested conflict list from [["CSCI320-01", "ENG210-01"]] into ["CSCI320-01", "ENG210-01"], so the answer value did not match the expected structure.
- **What you changed:** Updated the system prompt to require schedule-conflict answers to preserve the exact nested-list structure returned by the tool's conflicts field. Each conflict must remain a pair of section IDs, and no conflicts must be represented by an empty list ([]).
- **Result on the rerun:** The agent returned the correctly nested value [["CSCI320-01", "ENG210-01"]]. All public cases passed the answer, tool, and budget checks after the change.

## 11. Save your results

Before saving, restart the kernel and run the whole notebook from top to bottom, so the outputs and `results.json` come from your final tools and policy. `results.json` holds your names, your config, the public-case records, your six cases, and their records. The step-by-step traces are left out to keep the file small.

In [34]:
# YOUR WORK 6 — partner names, then save
PARTNERS = ["DENNIS", "RUBIN"]
assert all(p.strip() and p != "TODO" for p in PARTNERS), "Add your names."

save_results("results.json", partners=PARTNERS, config=check_config(CONFIG),
             public_records=public_records, my_cases=MY_CASES, my_records=my_records)

Saved results.json


## 12. Use of coding or AI assistants

**YOUR WORK 7:** Briefly state whether you used a coding or AI assistant. If you did, describe what you asked it to help you understand or debug, what suggestions you accepted or rejected, and how you verified the submitted work. If you did not use one, state that directly.

**Your answer:** I used AI for coding assistance while implementing and reviewing the Riverview College degree-planning tools. I reviewed the suggested logic in the context of the assignment's data and requirements. Most of the logic was sound and correctly handled the conditions of the tools. One corrction I needed to make when reviewing the check_eligibility code was that it didn't handle the case where a student got an F on a course and is currently retaking it. Afterwards, all public cases for the  tools passed. 

For the evaluation portion, I wrote the three entries from my own traces and asked it to review them. It suggested being more specific about which checks failed, quoting the added policy rules, and clarifying whether Change 2 and the courses_csci320_eng210 failure were related or not. I adopted these suggestions into the log, and stated that the failure could either be because the change caused the agent to design its answer differently, or because the agent is nondeterministic so it may be run-to-run variatiion. The assistant never saw my agent’s traces, only my descriptions of them.

I also used AI to help understand and debug problems involving setup and getting the model to work.

## Submission checklist

- [X] Both partners' names are in YOUR WORK 6.
- [X] `check_eligibility` and `schedule_conflicts` pass every public test.
- [X] Every public case passes.
- [X] Six cases, one per category, and your agent passes them.
- [X] The iteration log has three complete entries.
- [X] The notebook was run from top to bottom, with its outputs saved, and `results.json` was written.
- [X] The AI-assistance disclosure is complete.
- [X] No API key appears in either file.